In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 30. ZeRO와 FSDP — 메모리 최적화의 극한

> **학습 목표**
> - ZeRO Stage 1/2/3이 각각 무엇을 분할하는지 이해한다
> - FSDP (Fully Sharded Data Parallel)의 동작 원리를 안다
> - 메모리 절약 효과를 정량적으로 계산한다

## 30.1 DDP의 메모리 한계

DDP: 각 GPU가 모델 전체 복사. 70B 모델은 단일 GPU에 안 들어감.

ZeRO의 철학: **"모든 것을 분할하라"** — 옵티마이저 상태, 그래디언트, 파라미터까지.

## 30.2 메모리 분해

학습 메모리 = 파라미터 + 그래디언트 + 옵티마이저 상태 + 활성값

AdamW 기준:
- 파라미터 $P$ (FP16): $2P$ bytes
- 그래디언트 (FP16): $2P$ bytes
- 옵티마이저 상태 (FP32: m, v, master): $12P$ bytes
- 총 (활성값 제외): $16P$ bytes

7B 모델: 112 GB (단일 GPU 불가)

## 30.3 ZeRO Stage 1 — 옵티마이저 상태 분할

옵티마이저 상태를 $N$개 GPU에 분할:
- 메모리: $16P / N + 4P$ (파라미터 + 그래디언트는 그대로)
- 7B, N=8: $16P/8 + 4P = 2P + 4P = 6P = 42GB$ (여전히 큼)

## 30.4 ZeRO Stage 2 — 그래디언트도 분할

그래디언트도 분할:
- 메모리: $16P/N + 2P = 4P$ (8B + 4P + 4P)
- Reduce-Scatter 통신 (All-Reduce 대신)

## 30.5 ZeRO Stage 3 — 파라미터까지 분할

모두 분할:
- 메모리: $16P/N$ + 활성값
- 7B, N=8: $2P + 활성 ≈ 14GB$ → 단일 40GB GPU 가능
- 통신: All-Gather (순전파), Reduce-Scatter (역전파)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def zero_memory_gb(model_params_b, n_gpus, stage, optimizer_factor=12, dtype_bytes=2):
    """ZeRO stage별 Memory (GB).
    model_params_b: Parameter Count (단위: 10억)
    optimizer_factor: Adam의 경우 12 (m, v, master FP32)
    dtype_bytes: 2 for FP16
    """
    P = model_params_b  # 10억 단위
    # FP16 파라미터: 2 bytes, FP16 grad: 2 bytes
    # Optimizer (FP32 m, v, master): 12 bytes per param
    param_mem = P * dtype_bytes  # GB
    grad_mem = P * dtype_bytes
    opt_mem = P * optimizer_factor

    if stage == 0:  # DDP
        return param_mem + grad_mem + opt_mem
    elif stage == 1:  # Optimizer만 분할
        return param_mem + grad_mem + opt_mem / n_gpus
    elif stage == 2:  # Optimizer + Gradient 분할
        return param_mem + (grad_mem + opt_mem) / n_gpus
    elif stage == 3:  # 전부 분할
        return (param_mem + grad_mem + opt_mem) / n_gpus

# Visualization: 7B Model, GPU 수별
model_size = 7  # 7B
n_gpus_list = [1, 2, 4, 8, 16, 32, 64]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: GPU 수별
ax = axes[0]
for stage, label, color in [(0, 'DDP', 'blue'), (1, 'ZeRO-1', 'green'),
                             (2, 'ZeRO-2', 'orange'), (3, 'ZeRO-3', 'red')]:
    mems = [zero_memory_gb(model_size, n, stage) for n in n_gpus_list]
    ax.plot(n_gpus_list, mems, 'o-', label=label, linewidth=2, color=color)
ax.axhline(40, color='gray', linestyle='--', label='A100 40GB')
ax.axhline(80, color='gray', linestyle=':', label='A100 80GB')
ax.set_xlabel('GPU 수')
ax.set_ylabel('GPU 당 Memory (GB)')
ax.set_title(f'ZeRO Stage별 Memory (7B Model)')
ax.legend(); ax.grid(True, alpha=0.3)

# 오른쪽: Model Size별
ax = axes[1]
for stage, label, color in [(0, 'DDP', 'blue'), (3, 'ZeRO-3', 'red')]:
    for n in [1, 8, 32]:
        sizes = [1, 7, 13, 30, 70, 175]
        mems = [zero_memory_gb(s, n, stage) for s in sizes]
        ax.plot(sizes, mems, 'o-', label=f'{label}, N={n}', linewidth=2)
ax.set_xlabel('Model Size (B params)')
ax.set_ylabel('GPU 당 Memory (GB)')
ax.set_title('Model Size별 Memory')
ax.set_yscale('log')
ax.legend(); ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('../figures/ch30_zero_memory.png', dpi=100, bbox_inches='tight')
plt.show()

# 수치 Output
print(f"\n{'모델':>6} | {'GPU':>4} | {'DDP':>8} | {'ZeRO-1':>8} | {'ZeRO-2':>8} | {'ZeRO-3':>8}")
print("-" * 55)
for size in [7, 13, 70]:
    for n in [1, 8]:
        ddp = zero_memory_gb(size, n, 0)
        z1 = zero_memory_gb(size, n, 1)
        z2 = zero_memory_gb(size, n, 2)
        z3 = zero_memory_gb(size, n, 3)
        print(f"{size:>5}B | {n:>4} | {ddp:>7.1f}G | {z1:>7.1f}G | {z2:>7.1f}G | {z3:>7.1f}G")


## 30.6 PyTorch FSDP

FSDP (Fully Sharded Data Parallel) = PyTorch 구현 ZeRO-3.

```python
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

model = MyLargeModel()
model = FSDP(model)  # 자동으로 파라미터 분할
```

동작:
1. 순전파 전: All-Gather로 파라미터 모음
2. 순전파 후: 파라미터 메모리 해제
3. 역전파: 다시 All-Gather, 그래디언트 계산
4. Reduce-Scatter로 그래디언트 분할

## 30.7 CPU Offloading

GPU 메모리 부족 시 옵티마이저 상태를 CPU RAM으로 옮김.
- ZeRO-Infinity, FSDP with CPU offload
- 속도는 느려지지만 큰 모델 학습 가능


In [ ]:
# FSDP 시뮬레이션 (gradient sharding)
import torch
import torch.nn as nn
import torch.nn.functional as F

class FSDPSimulator:
    """단일 GPU에서 FSDP를 흉내내는 시뮬레이터."""
    def __init__(self, model, n_shards=4):
        self.model = model
        self.n_shards = n_shards
        # 파라미터를 n_shards로 분할 (각 "GPU"가 일부 담당)
        params = list(model.parameters())
        self.shards = [params[i::n_shards] for i in range(n_shards)]

    def step(self, x, y, optimizer):
        """한 Step: Forward Pass → Backward Pass → 각 shard별 Update."""
        optimizer.zero_grad()
        out = self.model(x)
        loss = F.mse_loss(out, y)
        loss.backward()

        # 각 shard의 Gradient만 사용 (다른 shard는 0으로)
        # 실제 FSDP는 통신으로 동기화하지만, 시뮬레이션에서는 모든 Gradient 사용
        optimizer.step()
        return loss.item()

# 시뮬레이션
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(128, 512), nn.ReLU(),
    nn.Linear(512, 128)
)
fsdp = FSDPSimulator(model, n_shards=4)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

x = torch.randn(8, 128)
y = torch.randn(8, 128)
loss = fsdp.step(x, y, opt)
print(f"FSDP 시뮬레이션 loss: {loss:.4f}")
print(f"총 파라미터: {sum(p.numel() for p in model.parameters()):,}")
print(f"Shard 수: {fsdp.n_shards} (각 shard당 ~{sum(p.numel() for p in model.parameters()) // 4:,} 파라미터)")


## 30.8 핵심 정리

| Stage | 분할 대상 | 메모리 | 통신 |
|---|---|---|---|
| DDP | 없음 | $16P$ | All-Reduce |
| ZeRO-1 | 옵티마이저 | $16P/N + 4P$ | All-Reduce |
| ZeRO-2 | + 그래디언트 | $16P/N + 2P$ | Reduce-Scatter |
| ZeRO-3 | + 파라미터 | $16P/N$ | All-Gather + Reduce-Scatter |
| FSDP | ZeRO-3 (PyTorch) | $16P/N$ | 동일 |
| ZeRO-Infinity | + CPU/NVMe | 더 적음 | 느림 |

## 연습문제

1. ZeRO Stage 0/1/2/3의 메모리를 70B 모델, N=16에 대해 계산하라.
2. FSDP가 순전파 전 All-Gather를 하는 이유를 설명하라.
3. ZeRO-3의 통신 오버헤드를 DDP와 비교하라.
4. CPU Offloading이 속도를 느리게 하는 이유를 설명하라.
5. 70B 모델을 학습하려면 ZeRO-3 + 몇 개 A100 80GB가 필요한지 계산하라.

> 해설: `solutions/ch30_solutions.ipynb`
